In [ ]:
import os

import httpx
import openai
from dotenv import find_dotenv, load_dotenv


load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 설정되어 있지 않습니다."

client = openai.OpenAI()
MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [ ]:
# --- 에이전트가 실제로 호출할 수 있는 함수들 ---


def get_popular_movies():
    """/movies 에서 인기 영화 목록을 가져옵니다."""
    return httpx.get(f"{BASE_URL}/movies", timeout=20).json()


def get_movie_details(id):
    """/movies/:id 에서 영화 상세 정보를 가져옵니다."""
    return httpx.get(f"{BASE_URL}/movies/{id}", timeout=20).json()


def get_movie_credits(id):
    """/movies/:id/credits 에서 출연진·제작진을 가져옵니다."""
    return httpx.get(f"{BASE_URL}/movies/{id}/credits", timeout=20).json()


FUNCTIONS = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [ ]:

PROMPT = """
I have the following functions in my movie system.

'get_popular_movies'   -> takes NO argument.        Returns currently popular movies.
'get_movie_details'    -> takes a movie id (number). Returns details of that movie.
'get_movie_credits'    -> takes a movie id (number). Returns the cast and crew of that movie.

Choose the single function that best answers the user's question.

Please answer ONLY with the function name and its arguments, nothing else.
Examples of valid answers:
get_popular_movies()
get_movie_details(550)
get_movie_credits(550)

Answer the following question:

{question}
"""

In [ ]:
def decide_function(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": PROMPT.format(question=question)},
        ],
    )
    return response.choices[0].message.content.strip()


for q in [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]:
    print(f"Q: {q}")
    print(f"-> {decide_function(q)}\n")

In [1]:
import re


def run_agent(question):
    """질문 -> 모델이 함수 선택 -> 실제 함수 호출까지 한 번에 수행합니다."""
    call = decide_function(question)
    print(f"Q: {question}")
    print(f"모델이 고른 함수: {call}")

    match = re.match(r"\s*(\w+)\s*\((.*?)\)\s*", call)
    if not match:
        raise ValueError(f"함수 호출을 파싱할 수 없습니다: {call!r}")

    name, raw_args = match.group(1), match.group(2).strip()
    func = FUNCTIONS[name]
    args = [int(raw_args)] if raw_args else []  

    return func(*args)


result = run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")
print("제목:", result["title"])
print("개요:", result["overview"][:120], "...")

NameError: name 'decide_function' is not defined